# TFG V1 — Optimized 1D window loader

This version keeps the one-dimensional search problem from V0 and starts separating the circuit into reusable stages. It also tests an auxiliary penalty register as an early way to distinguish valid and invalid windows.

## Problem model

The grid is a binary 1D array. A candidate index `i` represents the start of a fixed-size window, and a window is valid when all loaded cells are free (`0`). The conceptual state is:

```text
sum_i |grid>|idx=i>|window_i>
```

## Circuit structure

1. Prepare a superposition over candidate indices.
2. Load each indexed window into the work register `m`.
3. Use auxiliary qubits to mark or penalize branches with nonzero window content.
4. Inspect the resulting amplitudes and the effect of postselection.

## Registers and data

- `n`: fixed 1D grid with free and occupied cells.
- `idx`: candidate window-start index.
- `m`: loaded candidate window.
- `f`, `p`: auxiliary qubits used for validity marking and penalty experiments.

## Purpose of this version

V1 is the first optimized prototype. It introduces Gray-style control-mask reuse and separates window loading from validity marking, but the penalty-register idea is kept as an experiment rather than the final amplification strategy.

In [ ]:
import time
import qiskit
import numpy as np

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.quantum_info import Statevector, SparsePauliOp, partial_trace
from qiskit.visualization import plot_bloch_multivector, plot_state_qsphere
from qiskit_aer import AerSimulator
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
# DISABLED: IBM Runtime import; using simulators only. from qiskit_ibm_runtime import EstimatorV2 as Estimator
# DISABLED: IBM Runtime import; using simulators only. from qiskit_ibm_runtime import SamplerV2 as Sampler
# DISABLED: IBM Runtime import; using simulators only. from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit.visualization import plot_histogram
print(qiskit.__version__)

In [ ]:
N = 4
M = 2
W = N - M + 1
k = int(np.ceil(np.log2(W))) if W > 1 else 1
print(f"Possible solutions W: {W}, qubits required to represent W positions -> k: {k}")

In [ ]:
# Optimized penalty-register prototype.
# Idea: keep the Gray-mask optimization on idx and, once the
# window is loaded in m, penalize branches with m != 000 by redirecting amplitude to an ancilla.
# After postselecting p = 0, nonzero windows disappear completely.

n_opt = QuantumRegister(N, "n")
idx_opt = QuantumRegister(k, "i")
m_opt = QuantumRegister(M, "m")
flag_opt = QuantumRegister(1, "f")
pen_opt = QuantumRegister(1, "p")
qc_opt = QuantumCircuit(n_opt, idx_opt, m_opt, flag_opt, pen_opt)

# ---------------------------
# Random initial state for n: between 0 and N-M occupied positions, all distinct
# rng_n = np.random.default_rng()
# num_occupied = int(rng_n.integers(0, N - M + 1))
# occupied_positions = sorted(rng_n.choice(N, size=num_occupied, replace=False).tolist()) if num_occupied > 0 else []

# for pos in occupied_positions:
#     qc_opt.x(n_opt[pos])

# print(f"Random number of occupied positions in n: {num_occupied}")
# print(f"Random occupied positions in n: {occupied_positions}")
qc_opt.x(n_opt[1])
# ---------------------------

penalty_theta = np.pi / 2
target_zero_window = "0" * M

# Prepare the uniform index superposition.
qc_opt.h(idx_opt)

# Gray order filtered to valid windows 0..W-1
order = [g for t in range(1 << k) if (g := (t ^ (t >> 1))) < W]

def zero_mask(i, k):
    # 1 => this idx[b] needs X to represent a control on 0
    return [1 if ((i >> b) & 1) == 0 else 0 for b in range(k)]

x_flips_opt = 0
current_mask = [0] * k

for i in order:
    target_mask = zero_mask(i, k)

    # Update only the mask bits that differ from the previous index.
    for b in range(k):
        if target_mask[b] != current_mask[b]:
            qc_opt.x(idx_opt[b])
            x_flips_opt += 1

    current_mask = target_mask

    # XOR the selected window onto m
    for j in range(M):
        controls = [idx_opt[b] for b in range(k)] + [n_opt[i + j]]
        qc_opt.mcx(controls, m_opt[j])

# Clear the mask at the end to leave idx as it was before the block
for b in range(k):
    if current_mask[b] == 1:
        qc_opt.x(idx_opt[b])
        x_flips_opt += 1

# Coherent penalty: mark m != 000 and redirect part of its amplitude to pen_opt
for q in m_opt:
    qc_opt.x(q)
qc_opt.mcx(list(m_opt), flag_opt[0])
qc_opt.x(flag_opt[0])
qc_opt.cry(2 * penalty_theta, flag_opt[0], pen_opt[0])


qc_opt.x(flag_opt[0])
qc_opt.mcx(list(m_opt), flag_opt[0])
for q in m_opt:
    qc_opt.x(q)

# X-cost reference for the original approach
x_flips_naive = 0
for i in range(W):
    z = sum(1 for b in range(k) if ((i >> b) & 1) == 0)
    x_flips_naive += 2 * z

print(f"Orden usado (Gray filtrado): {order}")
print(f"X en enfoque original: {x_flips_naive}")
print(f"X en enfoque optimizado: {x_flips_opt}")
print(f"Penalizacion sobre m != {target_zero_window}: theta = {penalty_theta:.4f} rad")
print(f"Factor de supervivencia al postseleccionar p=0: cos(theta)^2 = {np.cos(penalty_theta) ** 2:.6f}")

qc_opt.draw(output='mpl')

In [ ]:
sv_opt = Statevector(qc_opt)
sv_opt.draw(output='latex', max_size=2**qc_opt.num_qubits, prefix="|\\psi\\rangle = ")

In [ ]:
# Analysis of the optimized circuit with penalty.
# Since ancilla p becomes entangled with nonzero windows, we no longer expect
# the reduced state on (idx, m) to be pure. Therefore we show total probabilities
# and probabilities conditioned on the unpenalized branch p = 0.

tol_n = 1e-10
total_q_opt = qc_opt.num_qubits
array_posiciones = None
prob_pen_0 = 0.0
prob_pen_1 = 0.0
aggregated = {}

for basis_idx, amp in enumerate(sv_opt.data):
    prob = float(np.abs(amp) ** 2)
    if prob <= tol_n:
        continue

    bits_full = format(basis_idx, f'0{total_q_opt}b')
    pen_bit = bits_full[0]
    flag_bit = bits_full[1]
    m_desc = bits_full[2:2 + M]
    i_desc = bits_full[2 + M:2 + M + k]
    n_desc = bits_full[-N:]

    ventana = m_desc[::-1]
    indices = i_desc
    n_asc = n_desc[::-1]

    if array_posiciones is None:
        array_posiciones = n_asc
    elif n_asc != array_posiciones:
        raise ValueError("The n qubits are not deterministic (there is no unique position array).")

    if flag_bit != '0':
        raise ValueError("La ancilla flag no quedo descomputada al final del circuito.")

    key = (indices, ventana)
    if key not in aggregated:
        aggregated[key] = {'p0': 0.0, 'p1': 0.0}

    if pen_bit == '0':
        aggregated[key]['p0'] += prob
        prob_pen_0 += prob
    else:
        aggregated[key]['p1'] += prob
        prob_pen_1 += prob

if array_posiciones is None:
    raise ValueError("Could not infer the position array from the statevector.")

print(f"Position array = {array_posiciones}\n")
print(f"Probabilidad de p = 0: {prob_pen_0:.6f}")
print(f"Probabilidad de p = 1: {prob_pen_1:.6f}\n")

filas = []
for (indices, ventana), probs in aggregated.items():
    idx_int = int(indices, 2)
    p_total = probs['p0'] + probs['p1']
    p_cond = probs['p0'] / prob_pen_0 if prob_pen_0 > tol_n else 0.0
    filas.append((idx_int, indices, ventana, p_total, probs['p0'], probs['p1'], p_cond))

filas.sort(key=lambda x: (x[0], x[2]))

# First W results: valid windows
for _, indices, ventana, p_total, p0, p1, p_cond in filas[:W]:
    print(
        f"Indices: {indices} || Ventana: {ventana} || "
        f"P_total={p_total:.6f} || P(p=0)={p0:.6f} || P(p=1)={p1:.6f} || P_cond(p=0)={p_cond:.6f}"
    )
    print()

# Remaining results (outside the valid window range)
if len(filas) > W:
    print("----------------------------------------")
    print("Remaining states (indices outside the valid window range):\n")
    for _, indices, ventana, p_total, p0, p1, p_cond in filas[W:]:
        print(
            f"Indices: {indices} || Ventana: {ventana} || "
            f"P_total={p_total:.6f} || P(p=0)={p0:.6f} || P(p=1)={p1:.6f} || P_cond(p=0)={p_cond:.6f}"
        )
        print()

In [ ]:
# Dynamic circuit: first measure p and only measure idx/m if p = 0.
# This implements a repeat-until-success attempt:
# - if p = 0, the state has collapsed to the valid subspace and we read idx
# - if p = 1, that attempt is discarded

shots = 4096

c_p_dyn = ClassicalRegister(1, 'c_p_dyn')
c_idx_dyn = ClassicalRegister(k, 'c_i_dyn')
c_m_dyn = ClassicalRegister(M, 'c_m_dyn')

qc_dynamic = qc_opt.copy()
qc_dynamic.add_register(c_p_dyn, c_idx_dyn, c_m_dyn)
qc_dynamic.measure(pen_opt, c_p_dyn)

with qc_dynamic.if_test((c_p_dyn[0], 0)):
    qc_dynamic.measure(idx_opt, c_idx_dyn)
    qc_dynamic.measure(m_opt, c_m_dyn)

backend = AerSimulator()
result_dyn = backend.run(qc_dynamic, shots=shots, memory=True).result()
counts_dyn = result_dyn.get_counts()
memory_dyn = result_dyn.get_memory()

accepted_counts = {}
accepted_shots = 0
rejected_shots = 0

for bitstring in memory_dyn:
    pen_bits, m_bits, idx_bits = bitstring.split()
    if pen_bits == '0':
        accepted_shots += 1
        window_bits = m_bits[::-1]
        key = (idx_bits, window_bits)
        accepted_counts[key] = accepted_counts.get(key, 0) + 1
    else:
        rejected_shots += 1

accepted_sorted = sorted(accepted_counts.items(), key=lambda x: x[1], reverse=True)

print(f"Shots totales: {shots}")
print(f"Intentos validos (p=0): {accepted_shots}")
print(f"Intentos descartados (p=1): {rejected_shots}")
print(f"Tasa empirica de exito por intento: {accepted_shots / shots:.6f}")
print(f"Tasa teorica de exito por intento: {prob_pen_0:.6f}")
print("\nIndices obtenidos solo en intentos validos:")

for (idx_bits, window_bits), count in accepted_sorted[:10]:
    print(
        f"idx={idx_bits} || window={window_bits} || "
        f"counts={count} || prob_cond~={count / accepted_shots:.6f}"
    )

qc_dynamic.draw(output='mpl')

In [ ]:
# Classical simulation of the repeat-until-success scheme.
# We directly use the theoretical distribution already computed:
# - each attempt has probability prob_pen_0 of returning p = 0
# - conditioned on p = 0, idx follows the p_cond distribution over valid rows

num_experiments = 1000
max_attempts = 10
rng = np.random.default_rng(12345)

valid_rows = [(idx_bits, window_bits, p_cond) for _, idx_bits, window_bits, _, _, _, p_cond in filas if p_cond > 0]
valid_labels = [(idx_bits, window_bits) for idx_bits, window_bits, _ in valid_rows]
valid_probs = np.array([p_cond for _, _, p_cond in valid_rows], dtype=float)
valid_probs = valid_probs / valid_probs.sum()

successes = 0
attempts_to_success = []
valid_idx_counts = {}

for _ in range(num_experiments):
    for attempt in range(1, max_attempts + 1):
        if rng.random() < prob_pen_0:
            successes += 1
            attempts_to_success.append(attempt)
            chosen = rng.choice(len(valid_labels), p=valid_probs)
            idx_bits, window_bits = valid_labels[chosen]
            key = (idx_bits, window_bits)
            valid_idx_counts[key] = valid_idx_counts.get(key, 0) + 1
            break

success_probability_emp = successes / num_experiments
success_probability_theory = 1 - (1 - prob_pen_0) ** max_attempts
avg_attempts_emp = np.mean(attempts_to_success) if attempts_to_success else float('nan')
avg_attempts_theory = sum(attempt * prob_pen_0 * (1 - prob_pen_0) ** (attempt - 1) for attempt in range(1, max_attempts + 1))
valid_idx_sorted = sorted(valid_idx_counts.items(), key=lambda x: x[1], reverse=True)

print(f"Experimentos independientes: {num_experiments}")
print(f"Maximo de intentos por experimento: {max_attempts}")
print(f"Empirical probability of finding a valid index: {success_probability_emp:.6f}")
print(f"Theoretical probability of finding a valid index: {success_probability_theory:.6f}")
print(f"Numero medio de intentos hasta exito: {avg_attempts_emp:.6f}")
print(f"Media teorica truncada de intentos: {avg_attempts_theory:.6f}")
print("\nFrequencies of valid indices found:")

for (idx_bits, window_bits), count in valid_idx_sorted:
    print(
        f"idx={idx_bits} || window={window_bits} || "
        f"counts={count} || prob~={count / successes:.6f}"
    )

In [ ]:
# Detailed traces of several repeat-until-success experiments.
# Each experiment runs attempts until a valid solution is found.
# The output shows the full sequence of results until the first success.

num_trace_experiments = 12
rng_trace = np.random.default_rng(2026)

print(f"Probabilidad de exito por intento: {prob_pen_0:.6f}\n")

for exp_id in range(1, num_trace_experiments + 1):
    print(f"Experimento {exp_id}")
    attempt = 0

    while True:
        attempt += 1

        if rng_trace.random() < prob_pen_0:
            chosen = rng_trace.choice(len(valid_labels), p=valid_probs)
            idx_bits, window_bits = valid_labels[chosen]
            print(
                f"  Attempt {attempt}: p=0 -> valid solution found, "
                f"idx={idx_bits}, window={window_bits}"
            )
            print(f"  Total de intentos hasta exito: {attempt}\n")
            break

        print(f"  Intento {attempt}: p=1 -> descartar y repetir")